In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/student-mat.csv', sep=';')
df_original = df.copy()  # para graficar el antes de la limpieza
n_inicial = len(df)
df.shape

**Pregunta asignada:** ¿Estudiar más horas se asocia a mejores notas?

Variables: `studytime` (ordinal: 1 = <2 h, 2 = 2–5 h, 3 = 5–10 h, 4 = >10 h semanales) y `G3` / `aprobado`.

# Parte 1: Limpieza

## 1. Nulos, duplicados y valores fuera de rango

In [ ]:
print('Nulos:', df.isna().sum().sum())
print('Duplicados:', df.duplicated().sum())

# Rangos según la descripción oficial del dataset (UCI)
rangos = {'age': (15, 22), 'Medu': (0, 4), 'Fedu': (0, 4), 'traveltime': (1, 4),
          'studytime': (1, 4), 'failures': (0, 4), 'famrel': (1, 5), 'freetime': (1, 5),
          'goout': (1, 5), 'Dalc': (1, 5), 'Walc': (1, 5), 'health': (1, 5),
          'absences': (0, 93), 'G1': (0, 20), 'G2': (0, 20), 'G3': (0, 20)}
for col, (lo, hi) in rangos.items():
    print(col, 'fuera de rango:', (~df[col].between(lo, hi)).sum())

## 2. Estudiantes con G3 = 0

In [ ]:
ceros = df[df['G3'] == 0]
print('Estudiantes con G3 = 0:', len(ceros))
print('De ellos con G2 = 0:', (ceros['G2'] == 0).sum())
print('De ellos con 0 ausencias:', (ceros['absences'] == 0).sum())
ceros[['G1', 'G2', 'G3', 'absences']].describe()

**Interpretación:** según la descripción oficial, G1 y G2 son las notas del 1er y 2do periodo y G3 la nota final del 3er periodo, por lo que G3 está fuertemente correlacionada con G1 y G2. Estos estudiantes sí tenían nota en G1 (y varios en G2), así que un 0 en G3 rompe esa relación: no parece un rendimiento real sino que **no se presentaron a la evaluación final o abandonaron el curso**. Que casi todos tengan 0 ausencias refuerza esto: dejaron de asistir y ya no se registraron faltas.

## 3. Decisión

Se **excluyen** los registros con G3 = 0: representan abandono, no una nota obtenida, y mantenerlos distorsiona la media y la fuerte correlación entre G1, G2 y G3 que señala la descripción oficial.

*Nota para el análisis posterior:* como G3 depende tanto de G1 y G2, si se construye un modelo para predecir G3 (o `aprobado`) conviene probarlo también **sin** G1 y G2. Es más difícil, pero más útil, porque permite anticipar el resultado antes de que existan esas notas.

In [ ]:
df = df[df['G3'] > 0].copy()

## 4. Registros inicio vs. final

In [ ]:
print('Registros iniciales:', n_inicial)
print('Registros para el análisis:', len(df))
print('Excluidos:', n_inicial - len(df))

## 5. Variable derivada `aprobado`

In [ ]:
df['aprobado'] = (df['G3'] >= 10).astype(int)
df['aprobado'].value_counts()

# Parte 2: Estadística descriptiva

## 1. Resumen de G3 y absences

In [ ]:
resumen = df[['G3', 'absences']].describe().T
resumen['IQR'] = resumen['75%'] - resumen['25%']
resumen['asimetria'] = df[['G3', 'absences']].skew()
resumen.rename(columns={'50%': 'mediana'})

## 2. Medida de tendencia central

- **G3:** sin los ceros, la distribución es casi simétrica (asimetría cercana a 0) y la media y la mediana quedan muy cerca, así que la **media** la representa bien.
- **absences:** tiene asimetría positiva fuerte (muchos estudiantes con pocas faltas y unos pocos con muchas), lo que arrastra la media hacia arriba. La **mediana** la representa mejor porque no se ve afectada por esos valores extremos.

## 3. Atípicos en absences (1.5 × IQR)

In [ ]:
q1, q3 = df['absences'].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atipicos = df[(df['absences'] < lim_inf) | (df['absences'] > lim_sup)]
print(f'Límites: [{lim_inf}, {lim_sup}]')
print('Atípicos:', len(atipicos))
print(atipicos['absences'].sort_values().values)

# Gráfico 3: boxplot de absences, los puntos fuera de los bigotes son los atípicos (1.5 × IQR)
fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot(df['absences'], orientation='horizontal', whis=1.5)
ax.set(title='Gráfico 3. Ausencias: atípicos por 1.5 × IQR', xlabel='Ausencias', yticks=[])
plt.show()

**¿Qué hacer con ellos?** Están dentro del rango válido (0 a 93), así que no son errores de captura: son estudiantes que de verdad faltaron mucho y son relevantes para entender el rendimiento. Los **conservamos**, y usamos la mediana al describir `absences` para que no distorsionen el análisis.

## 4. Porcentaje de aprobados

In [ ]:
print(f"Aprobados: {df['aprobado'].mean() * 100:.1f}%")

## 5. Correlación entre G1, G2 y G3

In [ ]:
corr = df[['G1', 'G2', 'G3']].corr()

# Gráfico 4: mapa de calor de la correlación
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(3), corr.columns)
ax.set_yticks(range(3), corr.columns)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im)
ax.set_title('Gráfico 4. Correlación entre G1, G2 y G3')
plt.show()
corr

**Interpretación:** las tres notas tienen correlación positiva y alta, y la más fuerte es entre G2 y G3 (periodos consecutivos). Esto confirma lo que dice la descripción oficial: a quien le va bien en los primeros periodos casi siempre le va bien al final. Por eso G1 y G2 predicen muy bien G3, pero un modelo que las use sirve poco para anticipar el resultado con tiempo.

## 6. Variables de la pregunta asignada: `studytime` y `G3`

In [ ]:
etiquetas = {1: '<2 h', 2: '2-5 h', 3: '5-10 h', 4: '>10 h'}

# studytime es ordinal -> tabla de frecuencias
frec = df['studytime'].value_counts().sort_index()
tabla = pd.DataFrame({'horas': frec.index.map(etiquetas), 'n': frec,
                      '%': (frec / len(df) * 100).round(1)})
print(tabla)
print('Categorías con pocos casos (<30):', tabla.loc[tabla['n'] < 30, 'horas'].tolist())

# G3 es numérica -> resumen numérico
df['G3'].describe()

**Pocos casos:** la categoría de más de 10 horas (`studytime = 4`) tiene solo 24 estudiantes (6.7 %), muy pocos frente a los 182 de 2–5 h. Sus promedios son menos confiables y hay que interpretarlos con cuidado.

# ¿Estudiar más horas se asocia a mejores notas?

In [ ]:
por_grupo = df.groupby('studytime').agg(n=('G3', 'size'), media_G3=('G3', 'mean'),
                                        mediana_G3=('G3', 'median'), pct_aprobados=('aprobado', 'mean'))
por_grupo['pct_aprobados'] = (por_grupo['pct_aprobados'] * 100).round(1)
por_grupo.index = por_grupo.index.map(etiquetas)
print(por_grupo.round(2))

# Spearman porque studytime es ordinal (= Pearson sobre los rangos, así no hace falta scipy)
spearman = df['studytime'].rank().corr(df['G3'].rank())
print('\nCorrelación de Spearman studytime vs G3:', round(spearman, 3))

In [ ]:
grupos = sorted(df['studytime'].unique())

# Gráfico 5: distribución de G3 por horas de estudio
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([df.loc[df['studytime'] == g, 'G3'] for g in grupos])
ax.set_xticks(range(1, len(grupos) + 1), [etiquetas[g] for g in grupos])
ax.set(title='Gráfico 5. G3 según horas de estudio semanales', xlabel='Horas de estudio', ylabel='G3')
plt.show()

# Gráfico 6: porcentaje de aprobados por horas de estudio
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(por_grupo.index, por_grupo['pct_aprobados'])
for i, (v, n) in enumerate(zip(por_grupo['pct_aprobados'], por_grupo['n'])):
    ax.text(i, v + 1, f'{v}% (n={n})', ha='center')
ax.set(title='Gráfico 6. % de aprobados según horas de estudio', xlabel='Horas de estudio',
       ylabel='% aprobados', ylim=(0, 110))
plt.show()

**Conclusión:** **sí hay una asociación positiva, pero débil.** Los que estudian 5–10 h y más de 10 h tienen mejor media de G3 (12.6 y 12.7) y más aprobados (83 %) que los que estudian menos de 5 h (media ≈ 11.1–11.5, 70–74 % de aprobados). Pero la relación no es escalonada: el grupo de 2–5 h está incluso un poco por debajo del de <2 h. La correlación de Spearman es solo **0.11**, y las cajas del Gráfico 5 se solapan mucho, así que las horas de estudio explican poco de las diferencias entre estudiantes. Además, el grupo de más de 10 h tiene solo 24 casos, y una asociación no prueba que estudiar más *cause* mejores notas.